# Spoke Fourier Analysis

중앙에서 바깥 방향으로 좁은 각도 영역에 길게 뻗는 spoke 형태 불량을 찾기 위한 Fourier 분석 notebook입니다.

입력 raw 데이터에는 아래 역할을 하는 칼럼이 필요합니다. `.csv`와 `.txt`를 모두 읽을 수 있고, comma와 tab separator를 모두 시도합니다. 실제 파일의 칼럼명이 다르면 `Column Mapping` 입력값에서 매핑할 수 있습니다.

```text
root lot id, wafer id, chip x position, chip y position, bin number
```

`DEFECT_BIN_NOS`에 입력한 bin 값만 defect로 봅니다. 나머지 bin도 버리지 않고 wafer 전체 map을 반지름 1인 원으로 정규화하는 데 사용합니다. 별도의 x/y pitch 입력은 받지 않고, 각 wafer의 occupied coordinate extent를 이용해 x/y aspect를 자동 보정합니다. raw data에 다른 칼럼이 많아도 매핑에 필요한 칼럼만 읽습니다.

선택된 raw 칼럼은 먼저 문자열로 읽고, 좌표 칼럼만 계산 시점에 숫자로 변환합니다. 따라서 `wafer_id`가 `10` 같은 숫자형이거나 `W01` 같은 문자형이어도 처리할 수 있습니다.

## 계산식

입력한 defect bin set을 `B`라고 하면 chip별 defect indicator는 아래와 같습니다.

$$
d_i = \mathbf{1}(bin\_no_i \in B)
$$

모든 반경 영역을 사용하고, theta 방향을 기본 360개 bin으로 나눠 angular defect-rate signal을 만듭니다.

$$
p(\theta_j) = \mathrm{mean}\{d_i \mid \theta_i \in \mathrm{bin}_j\}
$$

평균 offset, 즉 전체 defect-rate 성분을 제거한 뒤 harmonic `k`의 Fourier amplitude를 계산합니다.

$$
A_k = 2\left|\frac{1}{N}\sum_j (p(\theta_j)-\bar{p})e^{-ik\theta_j}\right|
$$

실제 spoke는 낮은 harmonic에서 강하고 sinc 형태로 부드럽게 감쇠하는 반면, 원형 ring의 격자 artifact는 전체 harmonic에 불규칙하게 퍼지는 경향을 이용합니다. 저주파와 broadband energy는 다음과 같습니다.

$$
E_{low}=\sum_{k=1}^{K_L}A_k^2, \qquad E_{broad}=\sum_{k=K_B}^{K_{max}}A_k^2, \qquad E_{total}=\sum_{k=1}^{K_{max}}A_k^2
$$

broadband amplitude의 median을 noise floor로 빼고 $A_k^+=\max(A_k-\mathrm{median}(A_{broad}),0)$를 만든 뒤, 각도 폭 `w`인 pulse의 이론적 template과 cosine similarity를 계산합니다. 가장 잘 맞는 폭은 자동 선택합니다.

$$
T_k(w)=\left|\frac{\sin(kw/2)}{k}\right|, \qquad C_{sinc}=\max_w\frac{\sum_k A_k^+T_k(w)}{\sqrt{\sum_k(A_k^+)^2}\sqrt{\sum_kT_k(w)^2}}
$$

인접 harmonic 사이의 불규칙한 spike는 spectral roughness로 감점합니다.

$$
Q_{rough}=\frac{\sum_k(A_{k+1}^+-A_k^+)^2}{\sum_k(A_k^+)^2+\epsilon}, \qquad M_{smooth}=\frac{1}{1+Q_{rough}}
$$

최종 대표 점수는 저주파 신호 크기, 저주파 집중도, sinc 유사도, smoothness와 broadband penalty를 결합합니다.

$$
\mathrm{spoke\_fourier\_signal}=\sqrt{E_{low}/2}\cdot\frac{E_{low}}{E_{total}+\epsilon}\cdot C_{sinc}\cdot M_{smooth}\cdot\left(1-\frac{E_{broad}}{E_{total}+\epsilon}\right)
$$

`signal_to_noise`는 low-band RMS를 broadband RMS로 나눈 내부 참고값입니다. `ANGULAR_BINS=360`이면 기본 경계는 low `1..72`, broadband `90..180`이며 사용자가 별도로 입력할 필요는 없습니다.

spoke에 의한 불량률은 추정 폭만큼의 최적 각도 구간 `S`를 찾고, 구간 바깥의 평균 불량률을 background로 차감해 계산합니다.

$$
r_{bg}=\frac{N_{defect,out}}{N_{chip,out}}, \qquad N_{spoke}=\max(0,N_{defect,S}-r_{bg}N_{chip,S})
$$

$$
\mathrm{spoke\_defect\_rate}=\frac{N_{spoke}}{N_{chip,total}}
$$

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy==1.26.4",
    "polars": "polars==1.14.0",
    "matplotlib": "matplotlib==3.9.2",
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

## User Inputs

`DEFECT_BIN_NOS`는 list 또는 scalar로 입력할 수 있습니다. 예를 들어 `DEFECT_BIN_NOS = 12`는 자동으로 `[12]`처럼 처리됩니다. `Column Mapping` 값에는 raw file 안의 실제 칼럼명을 넣습니다. 검증용 wafer 선택값은 분석 실행 후의 `Selected Wafer Validation` 셀에서 별도로 입력합니다.

In [ ]:
from pathlib import Path

INPUT_FILE = Path("input.txt")
OUTPUT_CSV = Path("spoke_fourier_output.csv")

# Column Mapping: raw file 안의 실제 칼럼명을 지정합니다.
ROOT_LOT_ID_COL = "root_lot_id"
WAFER_ID_COL = "wafer_id"
CHIP_X_COL = "chip_x_pos"
CHIP_Y_COL = "chip_y_pos"
BIN_NO_COL = "bin_no"

# list 또는 scalar 모두 허용합니다. 예: [12, 13] 또는 12
DEFECT_BIN_NOS = [12]

ANGULAR_BINS = 360
MIN_CHIPS = 20

## Run

아래 셀은 `spoke_fourier.py`의 backend 로직을 실행합니다. 결과는 `spoke_fourier_signal` 기준으로 정렬됩니다.

In [ ]:
import importlib
import spoke_fourier

importlib.reload(spoke_fourier)
from spoke_fourier import SpokeConfig, run_spoke_fourier

config = SpokeConfig(
    input_path=INPUT_FILE,
    defect_bin_nos=DEFECT_BIN_NOS,
    output_csv=OUTPUT_CSV,
    group_cols=(ROOT_LOT_ID_COL, WAFER_ID_COL),
    x_col=CHIP_X_COL,
    y_col=CHIP_Y_COL,
    bin_col=BIN_NO_COL,
    angular_bins=ANGULAR_BINS,
    min_chips=MIN_CHIPS,
)

run = run_spoke_fourier(config, write_csv=True)
result = run.result
analysis_df = run.analysis_df

print(f"defect bin_no values: {run.defect_bin_nos}")
print(f"saved: {OUTPUT_CSV}")
print(f"result rows={result.height:,}, columns={len(result.columns):,}")
print(f"analysis_df rows={analysis_df.height:,}, columns={len(analysis_df.columns):,}")
result.head(20)

## Result Columns

- `defect_rate`: 입력한 defect bin의 wafer 전체 chip 비율입니다.
- `spoke_defect_rate`: 검출된 spoke 각도 구간에서 background를 차감한 spoke 기인 추정 불량률입니다.
- `spoke_defect_chip_count_estimate`: spoke에 기인한 것으로 추정되는 background 보정 chip 수입니다.
- `spoke_fourier_signal`: 저주파 집중도, sinc 유사도, spectrum smoothness와 broadband penalty가 반영된 대표 점수입니다.
- `low_freq_fourier_signal`: low-frequency band의 Fourier energy 크기입니다.
- `low_freq_energy_ratio`: 전체 Fourier energy 중 low-frequency band 비율입니다.
- `sinc_similarity`: 실제 spectrum과 가장 잘 맞는 angular pulse sinc template의 유사도입니다.
- `estimated_spoke_width_deg`: 가장 잘 맞는 sinc template의 추정 각도 폭입니다.
- `broadband_energy_ratio`: 전체 frequency에 퍼진 불규칙 signal을 나타내는 진단값입니다.
- `signal_to_noise`: low-band RMS를 broadband RMS로 나눈 내부 참고값입니다.

In [ ]:
summary_cols = [
    *config.group_cols,
    "defect_rate",
    "spoke_defect_rate",
    "spoke_defect_chip_count_estimate",
    "spoke_fourier_signal",
    "low_freq_fourier_signal",
    "low_freq_energy_ratio",
    "sinc_similarity",
    "estimated_spoke_width_deg",
    "spoke_theta_center_deg",
    "spectral_smoothness",
    "broadband_energy_ratio",
    "dominant_harmonic",
    "signal_to_noise",
    "theta_bins_with_chips",
    "theta_coverage",
    "total_chip_count",
    "defect_chip_count",
]
result.select([col for col in summary_cols if col in result.columns]).head(30)

## Selected Wafer Validation

전체 wafer 분석 입력과 분리된 검증용 선택값입니다. `(root_lot_id, wafer_id)` 형태로 입력합니다. wafer id는 문자열 기준으로 비교하므로 `10`과 `"10"`은 같은 값으로 처리되고, 파일 값이 `W01`이면 `"W01"`로 입력합니다.

In [ ]:
# 검증할 wafer 하나를 선택합니다. 예: ("ABCDE", 10), ("ABCDE", "W01")
WAFER_TO_PLOT = ("ABCDE", 10)

## Selected Wafer Outputs

선택한 wafer 하나에 대해 전체 chip map, theta별 defect rate, 계산 가능한 모든 harmonic amplitude를 확인합니다. `ANGULAR_BINS = 360`이면 harmonic 1부터 180까지 계산합니다.

In [ ]:
import polars as pl
from spoke_fourier import (
    build_wafer_theta_signal,
    compute_harmonic_spectrum,
    max_calculable_harmonic,
    resolve_spectrum_bands,
)

theta_signal_df = build_wafer_theta_signal(
    analysis_df,
    WAFER_TO_PLOT,
    group_cols=config.group_cols,
    angular_bins=ANGULAR_BINS,
)

harmonic_spectrum_df = compute_harmonic_spectrum(
    theta_signal_df,
    angular_bins=ANGULAR_BINS,
    max_harmonic=max_calculable_harmonic(ANGULAR_BINS),
)

low_freq_max_harmonic, broadband_min_harmonic = resolve_spectrum_bands(
    angular_bins=ANGULAR_BINS,
    low_freq_max_harmonic=config.low_freq_max_harmonic,
    broadband_min_harmonic=config.broadband_min_harmonic,
)

selected_condition = pl.lit(True)
for column, value in zip(config.group_cols, WAFER_TO_PLOT):
    selected_condition = selected_condition & (
        pl.col(column).cast(pl.Utf8, strict=False).str.strip_chars() == str(value).strip()
    )
selected_result = result.filter(selected_condition)

print(f"selected wafer: {WAFER_TO_PLOT}")
print(f"theta bins with chips: {theta_signal_df.height:,}")
selected_result

## Selected Wafer Map

선택한 wafer의 전체 chip map입니다. `DEFECT_BIN_NOS`에 해당하는 chip은 검은색, 나머지 chip은 흰색으로 표시합니다. chip 좌표에서 x/y pitch를 추정해 직사각형들이 빈 공간 없이 맞닿도록 그립니다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle, Patch, Rectangle
from spoke_fourier import build_wafer_map_data

wafer_map = build_wafer_map_data(
    analysis_df,
    WAFER_TO_PLOT,
    group_cols=config.group_cols,
    x_col=config.x_col,
    y_col=config.y_col,
)
wafer_map_df = wafer_map.cells

rectangles = [
    Rectangle(
        (map_x - wafer_map.chip_width / 2, map_y - wafer_map.chip_height / 2),
        wafer_map.chip_width,
        wafer_map.chip_height,
    )
    for map_x, map_y in wafer_map_df.select("map_x", "map_y").iter_rows()
]
facecolors = [
    "black" if is_defect else "white"
    for is_defect in wafer_map_df.get_column("is_defect").to_list()
]

fig, ax = plt.subplots(figsize=(8.5, 7))
chip_collection = PatchCollection(
    rectangles,
    facecolors=facecolors,
    edgecolors="#b8b8b8",
    linewidths=0.2,
    antialiased=False,
)
ax.add_collection(chip_collection)
ax.add_patch(Circle((0, 0), wafer_map.wafer_radius, fill=False, edgecolor="black", linewidth=1.0))

plot_limit = wafer_map.wafer_radius * 1.04
ax.set_xlim(-plot_limit, plot_limit)
ax.set_ylim(-plot_limit, plot_limit)
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Wafer map: {WAFER_TO_PLOT}")
ax.axis("off")
ax.legend(
    handles=[
        Patch(facecolor="black", edgecolor="black", label=f"selected bin: {list(run.defect_bin_nos)}"),
        Patch(facecolor="white", edgecolor="#888888", label="other bins"),
    ],
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    frameon=False,
)
fig.tight_layout()
plt.show()

wafer_map_df

In [ ]:
import math
import matplotlib.pyplot as plt
from spoke_fourier import sinc_template_amplitudes

dominant_harmonic = None
estimated_spoke_width_deg = float("nan")
sinc_template_scale = 0.0
spoke_theta_center_rad = float("nan")
spoke_sector_width_deg = float("nan")
if selected_result.height:
    dominant_harmonic = selected_result.get_column("dominant_harmonic").item()
    estimated_spoke_width_deg = selected_result.get_column("estimated_spoke_width_deg").item()
    sinc_template_scale = selected_result.get_column("sinc_template_scale").item()
    spoke_theta_center_rad = selected_result.get_column("spoke_theta_center_rad").item()
    spoke_sector_width_deg = selected_result.get_column("spoke_sector_width_deg").item()

if math.isfinite(estimated_spoke_width_deg):
    matched_sinc = sinc_template_amplitudes(
        harmonic_spectrum_df.get_column("harmonic"),
        estimated_spoke_width_deg,
    ) * sinc_template_scale
else:
    matched_sinc = [float("nan")] * harmonic_spectrum_df.height
harmonic_spectrum_df = harmonic_spectrum_df.with_columns(
    pl.Series("matched_sinc_amplitude", matched_sinc)
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    theta_signal_df.get_column("theta_rad"),
    theta_signal_df.get_column("defect_rate"),
    marker="o",
    markersize=2,
    linewidth=1,
    label="defect rate",
)
if math.isfinite(spoke_theta_center_rad) and math.isfinite(spoke_sector_width_deg):
    half_width = math.radians(spoke_sector_width_deg) / 2.0
    sector_left = spoke_theta_center_rad - half_width
    sector_right = spoke_theta_center_rad + half_width
    sector_spans = [(sector_left, sector_right)]
    if sector_left < -math.pi:
        sector_spans = [(sector_left + 2 * math.pi, math.pi), (-math.pi, sector_right)]
    elif sector_right > math.pi:
        sector_spans = [(sector_left, math.pi), (-math.pi, sector_right - 2 * math.pi)]
    for index, (left, right) in enumerate(sector_spans):
        ax.axvspan(left, right, color="black", alpha=0.10, label="detected spoke sector" if index == 0 else None)
    ax.axvline(spoke_theta_center_rad, color="black", linestyle="--", linewidth=1)
ax.set_title(f"Defect rate vs theta: {WAFER_TO_PLOT}")
ax.set_xlabel("theta [rad]")
ax.set_ylabel("defect rate in theta bin")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(
    harmonic_spectrum_df.get_column("harmonic"),
    harmonic_spectrum_df.get_column("amplitude"),
    width=0.9,
    color="#4c78a8",
    label="measured amplitude",
)
ax.axvspan(1, low_freq_max_harmonic, color="#59a14f", alpha=0.12, label=f"low band: 1-{low_freq_max_harmonic}")
ax.axvspan(
    broadband_min_harmonic,
    max_calculable_harmonic(ANGULAR_BINS),
    color="#9c9c9c",
    alpha=0.14,
    label=f"broadband penalty: {broadband_min_harmonic}-{max_calculable_harmonic(ANGULAR_BINS)}",
)
if math.isfinite(estimated_spoke_width_deg):
    ax.plot(
        harmonic_spectrum_df.get_column("harmonic"),
        harmonic_spectrum_df.get_column("matched_sinc_amplitude"),
        color="#f28e2b",
        linewidth=2,
        label=f"matched sinc: {estimated_spoke_width_deg:.1f} deg",
    )
if dominant_harmonic is not None:
    ax.axvline(dominant_harmonic, color="red", linestyle="--", linewidth=1, label=f"dominant {dominant_harmonic}")
ax.set_title(f"Fourier amplitude spectrum: {WAFER_TO_PLOT}")
ax.set_xlabel("반복차수 (한 바퀴당 반복 횟수)")
ax.set_ylabel("amplitude")
ax.set_xlim(0, max_calculable_harmonic(ANGULAR_BINS) + 1)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

harmonic_spectrum_df.head(30)